<a href="https://colab.research.google.com/github/DarthCoder501/GAAP/blob/main/Copy_of_V1_PyTorch_Impressions_Model_FineTuned_(SMOTE_%26_HyperParameter).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 20.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import functional as F
import torch.backends.cudnn as cudnn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, classification_report, f1_score, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek
from sklearn.model_selection import StratifiedKFold
import optuna
import warnings
import random
import os
warnings.filterwarnings('ignore')

In [ ]:
# Set random seeds so we get the same results every time we run the code
SEED = 42

def set_seeds(seed=SEED):
    """Comprehensive seed setting for full reproducibility"""
    # Python random
    random.seed(seed)
    # NumPy
    np.random.seed(seed)
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU
    # PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # IMPORTANT: Changed from True to False
    torch.use_deterministic_algorithms(True, warn_only=True)  # Added for PyTorch 1.12+
    # Environment variables
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CUDA reproducibility
    # For transformers library
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Apply the seed & device settings
set_seeds(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load data
train = pd.read_csv("/content/Impressions Train Chronological Data.csv", parse_dates=["note_DATETIME"])
test = pd.read_csv("/content/Impressions Test Chronological Data.csv", parse_dates=["note_DATETIME"])

In [ ]:
# Load data
#train = pd.read_csv("/home/jpeguero/GAAP/data/Train Chronological (1).csv", parse_dates=["note_DATETIME"])
#test = pd.read_csv("/home/jpeguero/GAAP/data/Impressions Chronological Data (1).csv", parse_dates=["note_DATETIME"])

In [ ]:
# Abbreviations dictionary

abbreviations = {
"MM": "millimeter", "CT": "computed tomography", "HCW": "healthcare Worker",
"BAC": "bronchioloalveolar carcinoma", "ED": "emergency department", "PT": "pacific time",
"MRI": "magnetic resonance imaging", "RN": "registered nurse",
"PACS": "picture archiving and communication system", "PET": "positron emission tomography",
"SVC": "superior vena cava", "CM": "centimeter", "RLL": "right lower lobe",
"RUL": "right upper lobe", "LAD": "left anterior descending artery", "TB": "tuberculosis",
"IPMT": "intraductal papillary mucinous tumor", "IVC": "inferior vena cava",
"PE": "pulmonary embolism", "PEs": "pulmonary embolisms", "FDG": "fluorodeoxyglucose",
"SFV": "superficial femoral vein", "DVT": "deep vein thrombosis",
"SMA": "superior mesenteric artery", "NSIP": "nonspecific interstitial pneumonia",
"SITU": "in its original place", "HR": "hour",
"4A": "the superior part of the left medial segment of the liver",
"PST": "pacific standard time", "ID": "identification",
"CTA": "computed tomography angiography", "NG": "nasogastric",
"IPMN": "intraductal papillary mucinous neoplasm", "UIP": "usual interstitial pneumonia",
"ER": "emergency room", "ARDS": "acute respiratory distress syndrome",
"MRN": "medical record number", "RV": "right ventricular",
"CHF": "congestive heart failure", "PEG": "percutaneous endoscopic gastrostomy",
"PICC": "peripherally inserted central catheter", "GI": "gastrointestinal",
"ASD": "atrial septal defect", "MR": "mitral regurgitation",
"EST": "eastern standard time", "CTs": "computed tomographies",
"3D": "three dimensional", "MAC": "mycobacterium avium complex",
"MICU": "medical intensive care unit", "MAI": "mycobacterium avium-intracellulare",
"PJP": "pneumocystis jirovecii pneumonia", "LIMA": "left internal mammary artery",
"LV": "left ventricle", "EGD": "esophagogastroduodenoscopy",
"PAU": "penetrating atherosclerotic ulcer", "VP": "ventriculoperitoneal",
"CSF": "cerebrospinal fluid", "HCC": "hepatocellular carcinoma",
"SABR": "stereotactic ablative radiotherapy", "ILD": "interstitial lung disease",
"IVP": "intravenous pyelogram", "MRCP": "magnetic resonance cholangiopancreatography",
"IV": "intravenous", "RCA": "right coronary artery", "COVID": "coronavirus disease",
"2D": "two dimensional", "SMV": "superior mesenteric vein",
"FNA": "fine needle aspiration", "BAL": "bronchoalveolar Lavage",
"AVMs": "arteriovenous malformations", "AVM": "arteriovenous malformation",
"MRA": "magnetic resonance angiography", "AP": "anteroposterior",
"MRIs": "magnetic resonance imaging", "COVID19": "coronavirus disease 2019",
"BHD": "birt-hogg-dube", "CTEPH": "chronic thromboembolic pulmonary hypertension",
"RML": "right middle lobe", "NGT": "nasogastric tube", "GE": "gastroesophageal",
"MDS": "myelodysplastic syndrome", "UVJ": "ureterovesical junction",
"ERCP": "endoscopic retrograde cholangiopancreatography", "OP": "organizing pneumonia",
"IJ": "internal jugular", "VSD": "ventricular septal defect",
"EMR": "electronic medical record", "TE": "tracheoesophageal",
"AV": "arteriovenous", "PAN": "polyarteritis nodosa", "III": "third",
"SLE": "systemic lupus erythematosus", "CTS": "computed tomographies",
"IPF": "idiopathic pulmonary fibrosis", "3MM": "three millimeters",
"4MM": "four millimeters", "PAPVR": "partial anomalous pulmonary venous return",
"ANCA": "antineutrophil cytoplasmic antibodies", "VQ": "ventilation-perfusion",
"PA": "pulmonary artery", "PCP": "pneumocystis pneumonia",
"CMV": "cytomegalovirus", "RVH": "right ventricular hypertrophy",
"TSH": "thyroid stimulating hormone", "CBD": "common bile duct",
"BNP": "brain natriuretic peptide", "16MM": "sixteen millimeters",
"NP": "nurse practitioner", "CVC": "central venous catheter",
"SVG": "saphenous vein graft", "PDA": "posterior descending artery",
"VIII": "eighth", "ICU": "intensive care unit",
"CPR": "cardiopulmonary resuscitation", "DAH": "diffuse alveolar hemorrhage",
"PAP": "pulmonary alveolar proteinosis", "II": "second",
"ENT": "ear, nose, and throat", "FNH": "focal nodular hyperplasia",
"LLL": "left lower lobe", "CTPA": "computed tomography pulmonary angiography",
"LA": "left atrium", "ABPA": "allergic bronchopulmonary aspergillosis",
"IMA": "inferior mesenteric artery", "RT": "right", "CCU": "coronary care unit",
"ALS": "amyotrophic lateral sclerosis", "LT": "left", "RCC": "renal cell carcinoma",
"AML": "angiomyolipoma", "HCG": "human chorionic gonadotropin",
"IJV": "internal jugular vein", "LE": "lower extremity", "ASAP": "as soon as possible",
"1L": "one liter", "IHSS": "Idiopathic hypertrophic subaortic stenosis",
"13MM": "thirteen millimeters", "PFO": "patent foramen ovale",
"CCA": "common carotid artery", "SCA": "subclavian artery",
"ANS": "anteromedial basal subsegmental artery", "IgG4": "Immunoglobulin G4",
"ICD": "implantable cardioverter-defibrillator", "T9": "ninth thoracic vertebrae",
"CVICU": "cardiovascular intensive care unit", "T12": "twelfth thoracic vertebra",
"L5": "fifth lumbar vertebra", "L1": "first lumbar vertebra",
"L3": "third lumbar vertebra", "T4": "fourth thoracic vertebrae",
"T5": "fifth thoracic vertebra", "T7": "seventh thoracic vertebra",
"T8": "eighth thoracic vertebra", "T10": "tenth thoracic vertebra",
"L2": "second lumbar vertebra", "8MM": "eight millimeters",
"T2": "second thoracic vertebra", "IUD": "intrauterine device",
"T3": "third thoracic vertebrae", "T6": "sixth thoracic vertebrae",
"C7": "seventh cervical vertebra", "S4": "fourth heart sound",
"T11": "eleventh thoracic vertebra", "L4": "fourth lumbar vertebra",
"T1": "first thoracic vertebra", "S1": "first heart sound",
"PAH": "pulmonary arterial hypertension", "S9": "ninth heart sound",
"IMH": "intramural hematoma", "VATS": "video-assisted thoracoscopic surgery",
"S2": "second heart sound", "LVAD": "left ventricular assist device",

}

In [ ]:
# Replace medical abbreviations with full terms
train["impressions_clean"] = train["impressions"].replace(abbreviations, regex=True)
test["impressions_clean"] = test["impressions"].replace(abbreviations, regex=True)

# Convert to lowercase and remove special characters
train["impressions_clean"] = train["impressions_clean"].str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True)
test["impressions_clean"] = test["impressions_clean"].str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True)

# Remove rows with missing or empty text
train = train.dropna(subset=["impressions_clean"])
test = test.dropna(subset=["impressions_clean"])
train = train[train["impressions_clean"].str.strip() != ""]
test = test[test["impressions_clean"].str.strip() != ""]

print(f"Train samples after cleaning: {len(train)}")
print(f"Test samples after cleaning: {len(test)}")

Train samples after cleaning: 4460
Test samples after cleaning: 1116


In [ ]:
# Load Bio_ClinicalBERT
print("Loading Bio_ClinicalBERT...")
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
bert_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").to(device)
bert_model.eval()

# Tokenize text
max_seq_length = 512 # Maximum number of words we'll consider per medical note
print("Tokenizing text...")


def tokenize_data(texts, tokenizer, max_length=512):

  encodings = tokenizer(
  texts, # List of all medical notes
  padding=True, # Make all texts the same length by adding padding
  truncation=True, # Cut off texts that are too long
  max_length=max_seq_length, # Maximum length allowed
  return_tensors="pt" # Return in PyTorch format
  )

  return encodings


train_encodings = tokenize_data(train["impressions_clean"].tolist(), tokenizer, max_seq_length)
test_encodings = tokenize_data(test["impressions_clean"].tolist(), tokenizer, max_seq_length)

Loading Bio_ClinicalBERT...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Tokenizing text...


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [ ]:
def extract_embeddings_pytorch(model, encodings, batch_size=32):
    """Extract CLS token embeddings in batches using PyTorch"""
    model.eval()
    all_embeddings = []

    input_ids = encodings['input_ids']
    attention_mask = encodings['attention_mask']
    num_samples = input_ids.size(0)
    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            end_idx = min(i + batch_size, num_samples)
            batch_input_ids = input_ids[i:end_idx].to(device)
            batch_attention_mask = attention_mask[i:end_idx].to(device)
            outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
            # Use CLS token (first token) embeddings
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(batch_embeddings)
            if (i // batch_size + 1) % 10 == 0:
                print(f"Processed {i + batch_size} / {num_samples} samples")
    return np.concatenate(all_embeddings, axis=0)

In [ ]:
print("Extracting embeddings...")
X_train = extract_embeddings_pytorch(bert_model, train_encodings, batch_size=32)
X_test = extract_embeddings_pytorch(bert_model, test_encodings, batch_size=32)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

Extracting embeddings...
Processed 320 / 4460 samples
Processed 640 / 4460 samples
Processed 960 / 4460 samples
Processed 1280 / 4460 samples
Processed 1600 / 4460 samples
Processed 1920 / 4460 samples
Processed 2240 / 4460 samples
Processed 2560 / 4460 samples
Processed 2880 / 4460 samples
Processed 3200 / 4460 samples
Processed 3520 / 4460 samples
Processed 3840 / 4460 samples
Processed 4160 / 4460 samples
Processed 4480 / 4460 samples
Processed 320 / 1116 samples
Processed 640 / 1116 samples
Processed 960 / 1116 samples
Shape of X_train: (4460, 768)
Shape of X_test: (1116, 768)


In [ ]:
# Define targets
targets = ["1_month_readmission", "6_month_readmission", "12_month_readmission", "pe_positive"]

# Check if target columns exist and handle missing values
print("Preparing target variables...")
available_targets = [t for t in targets if t in train.columns and t in test.columns]
print(f"Available targets: {available_targets}")

if not available_targets:
    print("No target columns found! Please check column names.")
    print("Train columns:", train.columns.tolist())
    print("Test columns:", test.columns.tolist())
else:
    y_train = train[available_targets].fillna(0).astype(int)
    y_test = test[available_targets].fillna(0).astype(int)

Preparing target variables...
Available targets: ['1_month_readmission', '6_month_readmission', '12_month_readmission', 'pe_positive']


In [ ]:
print("Target distribution in training data:")
for target in available_targets:
    pos_count = y_train[target].sum()
    total_count = len(y_train)
    print(f"{target}: {pos_count}/{total_count} ({pos_count/total_count*100:.1f}% positive)")

Target distribution in training data:
1_month_readmission: 201/4460 (4.5% positive)
6_month_readmission: 593/4460 (13.3% positive)
12_month_readmission: 813/4460 (18.2% positive)
pe_positive: 1096/4460 (24.6% positive)


In [ ]:
print("Target distribution in test data:")
for target in available_targets:
    pos_count = y_test[target].sum()
    total_count = len(y_test)
    print(f"{target}: {pos_count}/{total_count} ({pos_count/total_count*100:.1f}% positive)")

Target distribution in test data:
1_month_readmission: 90/1116 (8.1% positive)
6_month_readmission: 204/1116 (18.3% positive)
12_month_readmission: 259/1116 (23.2% positive)
pe_positive: 226/1116 (20.3% positive)


In [ ]:
# Seeded Data Loaders
def create_reproducible_dataloader(dataset, batch_size, shuffle=True, num_workers=0):
    """Create reproducible DataLoader"""

    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=False if num_workers == 0 else True
    )

In [ ]:
# Advanced Sampling Strategy
def apply_advanced_sampling(X, y, target_name, seed=SEED):
    """Apply sampling with consistent seeding"""
    pos_count = np.sum(y == 1)
    neg_count = np.sum(y == 0)
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else 100

    if imbalance_ratio > 20:  # Severe imbalance
        k_neighbors = min(5, pos_count - 1)
        if k_neighbors < 1:
            return X, y
        sampler = BorderlineSMOTE(
            sampling_strategy=0.3,
            random_state=seed,  # Consistent seed
            k_neighbors=k_neighbors
        )
    elif imbalance_ratio > 10:  # Moderate imbalance
        n_neighbors = min(5, pos_count - 1)
        if n_neighbors < 1:
            return X, y
        sampler = ADASYN(
            sampling_strategy=0.4,
            random_state=seed,  # Consistent seed
            n_neighbors=n_neighbors
        )
    else:  # Balanced case
        return X, y

    try:
        return sampler.fit_resample(X, y)
    except Exception as e:
        print(f"Sampling failed for {target_name}: {e}")
        return X, y

# Apply sampling to each target
X_train_balanced = {}
y_train_balanced = {}

for target in available_targets:
    y_target = y_train[target].values
    if len(np.unique(y_target)) > 1:
        X_bal, y_bal = apply_advanced_sampling(X_train, y_target, target)
        X_train_balanced[target] = X_bal
        y_train_balanced[target] = y_bal
    else:
        X_train_balanced[target] = X_train
        y_train_balanced[target] = y_target

# Create consistent dataset sizes
min_size = min(len(X_train_balanced[t]) for t in available_targets)
X_train_final = X_train_balanced[available_targets[0]][:min_size]
y_train_final = [y_train_balanced[t][:min_size] for t in available_targets]



In [ ]:
# Model Architecture
class BalancedF1ModelV2(nn.Module):
    def __init__(self, input_dim, num_targets, dropout_rate=0.3, hidden_dims=[768, 384, 256], activation=nn.ReLU):
        super().__init__()
        self.num_targets = num_targets
        dim1, dim2, dim3 = hidden_dims

        # Shared layers with residual connections
        self.shared = nn.Sequential(
            nn.Linear(input_dim, dim1),
            nn.BatchNorm1d(dim1),
            activation(),
            nn.Dropout(dropout_rate),

            nn.Linear(dim1, dim2),
            nn.BatchNorm1d(dim2),
            activation(),
            nn.Dropout(dropout_rate),

            nn.Linear(dim2, dim3),
            nn.BatchNorm1d(dim3),
            activation(),
            nn.Dropout(dropout_rate * 0.7)
        )

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(dim3, dim3),
            nn.Tanh(),
            nn.Linear(dim3, dim3),
            nn.Softmax(dim=1)
        )

        # Task-specific branches
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim3, dim3 // 2),
                nn.BatchNorm1d(dim3 // 2),
                activation(),
                nn.Dropout(dropout_rate * 0.5),

                nn.Linear(dim3 // 2, dim3 // 4),
                nn.BatchNorm1d(dim3 // 4),
                activation(),
                nn.Dropout(dropout_rate * 0.3),

                nn.Linear(dim3 // 4, 1),
                nn.Sigmoid()
            ) for _ in range(num_targets)
        ])

        # Residual projections
        self.residual_proj1 = nn.Sequential(
            nn.Linear(dim1, dim2),  # Project from dim1 to dim2
            nn.BatchNorm1d(dim2),
            activation()
        )

    def forward(self, x):
        # Shared features
        x1 = self.shared[:4](x)  # First shared block (output dim1)
        x2 = self.shared[4:8](x1)  # Second shared block (output dim2)

        # Residual connection - Project x1 to dim2 before adding to x2
        x1_proj = self.residual_proj1(x1)
        x_res = x2 + x1_proj

        x3 = self.shared[8:](x_res)  # Third shared block (expects dim2 input, outputs dim3)

        # Attention
        attn = self.attention(x3)
        x_attended = x3 * attn

        # Task branches
        outputs = [branch(x_attended) for branch in self.branches]
        return outputs if len(outputs) > 1 else outputs[0]

In [ ]:
# Enhanced Loss Function
class BalancedF1LossEnhanced(nn.Module):
    def __init__(self, pos_weight=1.0, alpha=0.5, beta=0.6, gamma=1.5, eps=1e-8, positive_emphasis=1.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.eps = eps
        self.positive_emphasis = positive_emphasis

    def focal_loss(self, y_pred, y_true):
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)
        ce_loss = F.binary_cross_entropy(y_pred, y_true, reduction='none')
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.pos_weight + (1 - y_true) * 1.0
        focal_weight = alpha_t * (1 - p_t).pow(self.gamma)
        return (focal_weight * ce_loss).mean()

    def balanced_f1_loss(self, y_pred, y_true):
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)

        tp = (y_true * y_pred).sum()
        fp = ((1 - y_true) * y_pred).sum()
        fn = (y_true * (1 - y_pred)).sum()
        precision_pos = tp / (tp + fp + self.eps)
        recall_pos = tp / (tp + fn + self.eps)
        f1_pos = 2 * (precision_pos * recall_pos) / (precision_pos + recall_pos + self.eps)

        tn = ((1 - y_true) * (1 - y_pred)).sum()
        precision_neg = tn / (tn + fn + self.eps)
        recall_neg = tn / (tn + fp + self.eps)
        f1_neg = 2 * (precision_neg * recall_neg) / (precision_neg + recall_neg + self.eps)

        balanced_f1 = (self.alpha * f1_pos * self.positive_emphasis + (1 - self.alpha) * f1_neg) / \
                     (self.alpha * self.positive_emphasis + (1 - self.alpha))
        return 1 - balanced_f1

    def forward(self, y_pred, y_true):
        f1_loss = self.balanced_f1_loss(y_pred, y_true)
        focal_loss = self.focal_loss(y_pred, y_true)
        return self.beta * f1_loss + (1 - self.beta) * focal_loss

In [ ]:
# Training & Validation
def train_epoch(model, dataloader, optimizer, loss_functions, device, scheduler=None):
    """Reproducible training epoch"""
    model.train()
    total_loss = 0

    for batch_data in dataloader:
        batch_x = batch_data[0].to(device)
        batch_y_list = [batch_data[i+1].to(device) for i in range(len(loss_functions))]

        optimizer.zero_grad()
        outputs = model(batch_x)
        if not isinstance(outputs, list):
            outputs = [outputs]

        loss = 0
        for i, (output, target, loss_fn) in enumerate(zip(outputs, batch_y_list, loss_functions)):
            loss += loss_fn(output.squeeze(), target.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    if scheduler:
        scheduler.step()
    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, loss_functions, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch_data in dataloader:
            batch_x = batch_data[0].to(device)
            batch_y_list = [batch_data[i+1].to(device) for i in range(len(loss_functions))]

            outputs = model(batch_x)
            if not isinstance(outputs, list):
                outputs = [outputs]
            batch_loss = 0
            for i, (output, target, loss_fn) in enumerate(zip(outputs, batch_y_list, loss_functions)):
                batch_loss += loss_fn(output.squeeze(), target.float())
            total_loss += batch_loss.item()

    return total_loss / len(dataloader)

class BalancedF1Callback:
    def __init__(self, model, val_data, target_names, device, patience=10):
        self.model = model
        self.X_val, self.y_val_list = val_data
        self.target_names = target_names
        self.device = device
        self.patience = patience
        self.best_f1 = 0
        self.counter = 0

    def __call__(self, epoch, train_loss, val_loss):
        self.model.eval()
        with torch.no_grad():
            X_val_tensor = torch.FloatTensor(self.X_val).to(self.device)
            outputs = self.model(X_val_tensor)
            if not isinstance(outputs, list):
                outputs = [outputs]

            total_f1 = 0
            for i, (output, y_true) in enumerate(zip(outputs, self.y_val_list)):
                y_pred = output.cpu().numpy().ravel()
                precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
                f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
                best_threshold = thresholds[np.argmax(f1_scores)] if len(thresholds) > 0 else 0.5
                y_pred_binary = (y_pred > best_threshold).astype(int)

                f1_pos = f1_score(y_true, y_pred_binary, pos_label=1)
                f1_neg = f1_score(y_true, y_pred_binary, pos_label=0)
                balanced_f1 = 0.8 * f1_pos + 0.2 * f1_neg
                total_f1 += balanced_f1

            avg_f1 = total_f1 / len(self.target_names)

            if avg_f1 > self.best_f1:
                self.best_f1 = avg_f1
                self.counter = 0
            else:
                self.counter += 1

            print(f"Epoch {epoch+1} - Balanced F1: {avg_f1:.4f} (Best: {self.best_f1:.4f})")
            return avg_f1, self.counter >= self.patience

In [ ]:
# Hyperparameter Optimization
def optimize_hyperparameters(X_train, y_train, X_test, y_test, target_names, seed=SEED):
    """Reproducible hyperparameter optimization"""

    def objective(trial):
        # Set seeds at the beginning of each trial
        set_seeds(seed + trial.number)  # Different seed per trial but deterministic

        params = {
            'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
            'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
            'hidden_dim1': trial.suggest_categorical('hidden_dim1', [512, 768]),
            'hidden_dim2': trial.suggest_categorical('hidden_dim2', [256, 384]),
            'pos_weight_alpha': trial.suggest_float('pos_weight_alpha', 0.3, 0.8),
        }

        # Apply sampling with consistent seeding
        X_train_balanced = {}
        y_train_balanced = {}
        for target in target_names:
            y_target = y_train[target].values
            if len(np.unique(y_target)) > 1:
                X_bal, y_bal = apply_advanced_sampling(X_train, y_target, target, seed)
                X_train_balanced[target] = X_bal
                y_train_balanced[target] = y_bal
            else:
                X_train_balanced[target] = X_train
                y_train_balanced[target] = y_target

        min_size = min(len(X_train_balanced[t]) for t in target_names)
        X_train_final = X_train_balanced[target_names[0]][:min_size]
        y_train_final = [y_train_balanced[t][:min_size] for t in target_names]

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train_final)
        y_train_tensors = [torch.FloatTensor(y) for y in y_train_final]
        X_test_tensor = torch.FloatTensor(X_test)
        y_test_tensors = [torch.FloatTensor(y_test[t].values) for t in target_names]

        # Create reproducible dataloaders
        train_dataset = TensorDataset(X_train_tensor, *y_train_tensors)
        train_loader = create_reproducible_dataloader(train_dataset, params['batch_size'], shuffle=True)
        val_dataset = TensorDataset(X_test_tensor, *y_test_tensors)
        val_loader = create_reproducible_dataloader(val_dataset, params['batch_size'], shuffle=False)

        # Initialize model with deterministic initialization
        model = BalancedF1ModelV2(
            input_dim=X_train.shape[1],
            num_targets=len(target_names),
            dropout_rate=params['dropout_rate'],
            hidden_dims=[params['hidden_dim1'], params['hidden_dim2'], params['hidden_dim2'] // 2]
        ).to(device)

        # Initialize loss functions
        loss_functions = []
        for i, target in enumerate(target_names):
            y_target = y_train[target].values
            pos_count = np.sum(y_target == 1)
            neg_count = np.sum(y_target == 0)
            pos_weight = min(neg_count / pos_count, 3.0) if pos_count > 0 else 1.0
            pos_weight = pos_weight if np.isfinite(pos_weight) else 1.0

            # Apply emphasis for readmission targets
            positive_emphasis = 2.0 if target in ['1_month_readmission', '6_month_readmission', '12_month_readmission'] else 1.0

            loss_functions.append(BalancedF1LossEnhanced(
                pos_weight=pos_weight * params['pos_weight_alpha'],
                alpha=0.5,
                beta=0.6,
                gamma=1.5,
                positive_emphasis=positive_emphasis
            ))

        optimizer = optim.AdamW(model.parameters(), lr=params['learning_rate'])
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[
                optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=5),
                optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)
            ],
            milestones=[5]
        )

        # Training loop with reproducible training
        for epoch in range(10):
            train_epoch(model, train_loader, optimizer, loss_functions, device, scheduler)

        # Calculate balanced F1 score
        model.eval()
        with torch.no_grad():
            outputs = model(X_test_tensor.to(device))
            if not isinstance(outputs, list):
                outputs = [outputs]

            balanced_f1 = 0
            for i, (output, target) in enumerate(zip(outputs, y_test_tensors)):
                y_pred = output.squeeze().cpu().numpy()
                y_true = target.cpu().numpy()
                if len(np.unique(y_true)) > 1:
                    f1_pos = f1_score(y_true, (y_pred > 0.5).astype(int), pos_label=1)
                    f1_neg = f1_score(y_true, (y_pred > 0.5).astype(int), pos_label=0)
                    balanced_f1 += (0.8 * f1_pos + 0.2 * f1_neg)
                else:
                    balanced_f1 += 0

        return balanced_f1 / len(target_names)

    # Create reproducible Optuna study
    sampler = optuna.samplers.TPESampler(seed=seed)  # Seeded sampler
    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    )

    study.optimize(objective, n_trials=30)
    return study.best_params

print("Starting hyperparameter optimization...")
best_params = optimize_hyperparameters(X_train, y_train, X_test, y_test, available_targets)
print("Best parameters found:", best_params)

[I 2025-07-19 01:20:58,089] A new study created in memory with name: no-name-47526080-3678-4c8d-b58d-d53e28fb57f3


Starting hyperparameter optimization...


[I 2025-07-19 01:21:36,901] Trial 0 finished with value: 0.43199894934119165 and parameters: {'dropout_rate': 0.249816047538945, 'learning_rate': 0.0007969454818643932, 'batch_size': 32, 'hidden_dim1': 512, 'hidden_dim2': 256, 'pos_weight_alpha': 0.6540362888980227}. Best is trial 0 with value: 0.43199894934119165.
[I 2025-07-19 01:22:14,961] Trial 1 finished with value: 0.42977625705889855 and parameters: {'dropout_rate': 0.10823379771832098, 'learning_rate': 0.0008706020878304854, 'batch_size': 32, 'hidden_dim1': 768, 'hidden_dim2': 256, 'pos_weight_alpha': 0.44561457009902095}. Best is trial 0 with value: 0.43199894934119165.
[I 2025-07-19 01:22:25,333] Trial 2 finished with value: 0.35303120446926817 and parameters: {'dropout_rate': 0.34474115788895177, 'learning_rate': 1.9010245319870364e-05, 'batch_size': 128, 'hidden_dim1': 512, 'hidden_dim2': 384, 'pos_weight_alpha': 0.32322520635999885}. Best is trial 0 with value: 0.43199894934119165.
[I 2025-07-19 01:22:35,728] Trial 3 finis

Best parameters found: {'dropout_rate': 0.20142832385590512, 'learning_rate': 0.00040172003100929143, 'batch_size': 32, 'hidden_dim1': 512, 'hidden_dim2': 256, 'pos_weight_alpha': 0.6643991973657172}


In [ ]:
"""Final Model Training"""

# Initialize final model with best parameters
model_final = BalancedF1ModelV2(
    input_dim=X_train.shape[1],
    num_targets=len(available_targets),
    dropout_rate=best_params['dropout_rate'],
    hidden_dims=[best_params['hidden_dim1'], best_params['hidden_dim2'], best_params['hidden_dim2'] // 2]
).to(device)

# Initialize enhanced loss functions
loss_functions_final = []
for i, target in enumerate(available_targets):
    y_target = y_train[target].values
    pos_count = np.sum(y_target == 1)
    neg_count = np.sum(y_target == 0)
    pos_weight = min(neg_count / pos_count, 3.0) if pos_count > 0 else 1.0
    pos_weight = pos_weight if np.isfinite(pos_weight) else 1.0

    # Apply emphasis for readmission targets
    positive_emphasis = 2.0 if target in ['1_month_readmission', '6_month_readmission', '12_month_readmission'] else 1.0

    loss_functions_final.append(BalancedF1LossEnhanced(
        pos_weight=pos_weight * best_params['pos_weight_alpha'],
        alpha=0.5,
        beta=0.6,
        gamma=1.5,
        positive_emphasis=positive_emphasis
    ))

# Initialize optimizer and scheduler
optimizer_final = optim.AdamW(model_final.parameters(), lr=best_params['learning_rate'])
scheduler_final = optim.lr_scheduler.SequentialLR(
    optimizer_final,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer_final, start_factor=0.01, total_iters=5),
        optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_final, T_0=10, T_mult=2, eta_min=1e-7)
    ],
    milestones=[5]
)

# Prepare data loaders
X_train_tensor = torch.FloatTensor(X_train_final)
y_train_tensors = [torch.FloatTensor(y) for y in y_train_final]
train_dataset = TensorDataset(X_train_tensor, *y_train_tensors)
train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensors = [torch.FloatTensor(y_test[t].values) for t in available_targets]
test_dataset = TensorDataset(X_test_tensor, *y_test_tensors)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

# Initialize callback
callback = BalancedF1Callback(
    model=model_final,
    val_data=(X_test, [y_test[t].values for t in available_targets]),
    target_names=available_targets,
    device=device,
    patience=15
)

# Training loop
num_epochs = 100
best_balanced_f1 = 0
train_losses = []
val_losses = []

print("Starting final training...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model_final, train_loader, optimizer_final, loss_functions_final, device, scheduler_final)
    val_loss = validate_epoch(model_final, test_loader, loss_functions_final, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    balanced_f1, stop_early = callback(epoch, train_loss, val_loss)

    if balanced_f1 > best_balanced_f1:
        best_balanced_f1 = balanced_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_final.state_dict(),
            'optimizer_state_dict': optimizer_final.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'balanced_f1': balanced_f1,
            'target_names': available_targets
        }, 'best_model_final.pth')

    if stop_early:
        print("Early stopping triggered")
        break

print("Training completed!")

Starting final training...
Epoch 1 - Balanced F1: 0.2706 (Best: 0.2706)
Epoch 2 - Balanced F1: 0.3644 (Best: 0.3644)
Epoch 3 - Balanced F1: 0.3887 (Best: 0.3887)
Epoch 4 - Balanced F1: 0.4005 (Best: 0.4005)
Epoch 5 - Balanced F1: 0.3936 (Best: 0.4005)
Epoch 6 - Balanced F1: 0.4556 (Best: 0.4556)
Epoch 7 - Balanced F1: 0.4405 (Best: 0.4556)
Epoch 8 - Balanced F1: 0.4271 (Best: 0.4556)
Epoch 9 - Balanced F1: 0.4501 (Best: 0.4556)
Epoch 10 - Balanced F1: 0.4546 (Best: 0.4556)
Epoch 11 - Balanced F1: 0.4015 (Best: 0.4556)
Epoch 12 - Balanced F1: 0.4178 (Best: 0.4556)
Epoch 13 - Balanced F1: 0.4178 (Best: 0.4556)
Epoch 14 - Balanced F1: 0.4291 (Best: 0.4556)
Epoch 15 - Balanced F1: 0.3911 (Best: 0.4556)
Epoch 16 - Balanced F1: 0.4227 (Best: 0.4556)
Epoch 17 - Balanced F1: 0.4505 (Best: 0.4556)
Epoch 18 - Balanced F1: 0.4187 (Best: 0.4556)
Epoch 19 - Balanced F1: 0.4100 (Best: 0.4556)
Epoch 20 - Balanced F1: 0.4245 (Best: 0.4556)
Epoch 21 - Balanced F1: 0.3867 (Best: 0.4556)
Early stopping t

In [ ]:
# Model Evaluation
class F1EnsemblePredictor:
    def __init__(self, model, target_names, device):
        self.model = model
        self.target_names = target_names
        self.device = device
        self.thresholds = {}

    def find_balanced_threshold(self, y_true, y_pred_proba):
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
        best_balanced_f1 = 0
        best_threshold = 0.5
        for i, threshold in enumerate(thresholds):
            y_pred_binary = (y_pred_proba >= threshold).astype(int)
            f1_pos = f1_score(y_true, y_pred_binary, pos_label=1, zero_division=0)
            f1_neg = f1_score(y_true, y_pred_binary, pos_label=0, zero_division=0)
            balanced_f1 = 0.8 * f1_pos + 0.2 * f1_neg
            if balanced_f1 > best_balanced_f1:
                best_balanced_f1 = balanced_f1
                best_threshold = threshold
        return best_threshold, best_balanced_f1

    def fit_thresholds(self, X_val, y_val_list):
        self.model.eval()
        with torch.no_grad():
            X_val_tensor = torch.FloatTensor(X_val).to(self.device)
            predictions = self.model(X_val_tensor)
            if not isinstance(predictions, list):
                predictions = [predictions]
            for i, (pred, y_true) in enumerate(zip(predictions, y_val_list)):
                y_pred = pred.cpu().numpy().ravel()
                threshold, balanced_f1 = self.find_balanced_threshold(y_true, y_pred)
                self.thresholds[self.target_names[i]] = threshold
                print(f"Optimal threshold for {self.target_names[i]}: {threshold:.3f} (Balanced F1: {balanced_f1:.3f})")

    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model(X_tensor)
            if not isinstance(predictions, list):
                predictions = [predictions]
            results = {}
            for i, target in enumerate(self.target_names):
                y_pred_proba = predictions[i].cpu().numpy().ravel()
                threshold = self.thresholds.get(target, 0.5)
                results[target] = {
                    'probabilities': y_pred_proba,
                    'predictions': (y_pred_proba >= threshold).astype(int),
                    'threshold': threshold
                }
            return results



In [ ]:
# Load best model
checkpoint = torch.load('best_model_final.pth')
model_final.load_state_dict(checkpoint['model_state_dict'])
model_final.eval()

BalancedF1ModelV2(
  (shared): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.20142832385590512, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.20142832385590512, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.14099982669913358, inplace=False)
  )
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Softmax(dim=1)
  )
  (branches): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=128,

In [ ]:
# Initialize predictor and evaluate
predictor = F1EnsemblePredictor(model_final, available_targets, device)
print("\nFitting optimal thresholds...")
predictor.fit_thresholds(X_test, [y_test[t].values for t in available_targets])

print("\nGenerating predictions...")
results = predictor.predict(X_test)


Fitting optimal thresholds...
Optimal threshold for 1_month_readmission: 0.306 (Balanced F1: 0.323)
Optimal threshold for 6_month_readmission: 0.336 (Balanced F1: 0.379)
Optimal threshold for 12_month_readmission: 0.368 (Balanced F1: 0.409)
Optimal threshold for pe_positive: 0.556 (Balanced F1: 0.829)

Generating predictions...


In [ ]:
# Print performance metrics
print("\n" + "="*80)
print("FINAL MODEL PERFORMANCE")
print("="*80)

final_metrics = {}
for target in available_targets:
    y_true = y_test[target].values
    y_pred = results[target]['predictions']
    y_proba = results[target]['probabilities']
    threshold = results[target]['threshold']

    # Calculate metrics
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    accuracy = (tp + tn) / len(y_true)
    precision_pos = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_pos = 2 * (precision_pos * recall_pos) / (precision_pos + recall_pos) if (precision_pos + recall_pos) > 0 else 0

    precision_neg = tn / (tn + fn) if (tn + fn) > 0 else 0
    recall_neg = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_neg = 2 * (precision_neg * recall_neg) / (precision_neg + recall_neg) if (precision_neg + recall_neg) > 0 else 0

    balanced_f1 = (f1_pos + f1_neg) / 2

    try:
        auc_score = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan
    except:
        auc_score = np.nan

    final_metrics[target] = {
        'auc_roc': auc_score,
        'accuracy': accuracy,
        'precision_pos': precision_pos,
        'recall_pos': recall_pos,
        'f1_pos': f1_pos,
        'precision_neg': precision_neg,
        'recall_neg': recall_neg,
        'f1_neg': f1_neg,
        'balanced_f1': balanced_f1,
        'threshold': threshold
    }

    print(f"\n{target} Results (Threshold: {threshold:.3f}):")
    print(f"AUC-ROC: {auc_score:.4f}" if not np.isnan(auc_score) else "AUC-ROC: N/A")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nPositive Class (1):")
    print(f"Precision: {precision_pos:.4f}, Recall: {recall_pos:.4f}, F1: {f1_pos:.4f}")
    print(f"\nNegative Class (0):")
    print(f"Precision: {precision_neg:.4f}, Recall: {recall_neg:.4f}, F1: {f1_neg:.4f}")
    print(f"\nBalanced F1: {balanced_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))


FINAL MODEL PERFORMANCE

1_month_readmission Results (Threshold: 0.306):
AUC-ROC: 0.5816
Accuracy: 0.8199

Positive Class (1):
Precision: 0.1419, Recall: 0.2444, F1: 0.1796

Negative Class (0):
Precision: 0.9292, Recall: 0.8704, F1: 0.8988

Balanced F1: 0.5392

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.87      0.90      1026
           1       0.14      0.24      0.18        90

    accuracy                           0.82      1116
   macro avg       0.54      0.56      0.54      1116
weighted avg       0.87      0.82      0.84      1116


6_month_readmission Results (Threshold: 0.336):
AUC-ROC: 0.5394
Accuracy: 0.5556

Positive Class (1):
Precision: 0.2137, Recall: 0.5343, F1: 0.3053

Negative Class (0):
Precision: 0.8432, Recall: 0.5603, F1: 0.6733

Balanced F1: 0.4893

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.56      0.67       912
           1     

In [ ]:
# Print summary statistics
print("\n" + "="*80)
print("OVERALL PERFORMANCE SUMMARY")
print("="*80)

avg_metrics = {
    'auc_roc': np.nanmean([final_metrics[t]['auc_roc'] for t in available_targets]),
    'accuracy': np.mean([final_metrics[t]['accuracy'] for t in available_targets]),
    'precision_pos': np.mean([final_metrics[t]['precision_pos'] for t in available_targets]),
    'recall_pos': np.mean([final_metrics[t]['recall_pos'] for t in available_targets]),
    'f1_pos': np.mean([final_metrics[t]['f1_pos'] for t in available_targets]),
    'precision_neg': np.mean([final_metrics[t]['precision_neg'] for t in available_targets]),
    'recall_neg': np.mean([final_metrics[t]['recall_neg'] for t in available_targets]),
    'f1_neg': np.mean([final_metrics[t]['f1_neg'] for t in available_targets]),
    'balanced_f1': np.mean([final_metrics[t]['balanced_f1'] for t in available_targets])
}

print(f"Average AUC-ROC: {avg_metrics['auc_roc']:.4f}")
print(f"Average Accuracy: {avg_metrics['accuracy']:.4f}")
print(f"\nPositive Class Averages:")
print(f"Precision: {avg_metrics['precision_pos']:.4f}, Recall: {avg_metrics['recall_pos']:.4f}, F1: {avg_metrics['f1_pos']:.4f}")
print(f"\nNegative Class Averages:")
print(f"Precision: {avg_metrics['precision_neg']:.4f}, Recall: {avg_metrics['recall_neg']:.4f}, F1: {avg_metrics['f1_neg']:.4f}")
print(f"\nOverall Balanced F1: {avg_metrics['balanced_f1']:.4f}")


OVERALL PERFORMANCE SUMMARY
Average AUC-ROC: 0.6546
Average Accuracy: 0.7007

Positive Class Averages:
Precision: 0.3507, Recall: 0.5458, F1: 0.4111

Negative Class Averages:
Precision: 0.8805, Recall: 0.7150, F1: 0.7807

Overall Balanced F1: 0.5959


In [ ]:
# Save the model
torch.save({
    'model_state_dict': model_final.state_dict(),
    'target_names': available_targets,
    'thresholds': predictor.thresholds # Save the optimal thresholds as well
}, 'final_model_V2.pth')

print("Model and thresholds saved to final_model_V2.pth")

Model and thresholds saved to final_model_V2.pth
